<a href="https://colab.research.google.com/github/SurgeryIMTM2627/Surgery2627/blob/main/Surgery_HIS_database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###**Import libraries and establish the connection**

In [ ]:
import sqlite3 #allows for working with SQLite databases
import pandas as pd #enables the visualization of tables and queries
from IPython.display import display

conn = sqlite3.connect(":memory:") #creates a temporary, in-memory database
conn.execute("PRAGMA foreign_keys = ON")

print("SQLite connection ready. Foreign keys are enabled.") #enables foreign key constraint checking

SQLite connection ready. Foreign keys are enabled.


###**Create the master tables**

In [ ]:
conn.executescript("""
CREATE TABLE professionals (
    id_professional INTEGER PRIMARY KEY,
    professional_name TEXT NOT NULL,
    professional_type TEXT NOT NULL
        CHECK (professional_type IN ('surgery', 'anaesthesia', 'nursing')),
    professional_category TEXT NOT NULL
        CHECK (professional_category IN ('Resident', 'Specialist', 'Consultant')),
    professional_email TEXT UNIQUE,
    professional_phone TEXT
);

CREATE TABLE operating_rooms (
    id_room INTEGER PRIMARY KEY,
    room_code TEXT NOT NULL UNIQUE,
    room_name TEXT NOT NULL,
    room_location TEXT NOT NULL,
    room_type TEXT NOT NULL
        CHECK (room_type IN ('urgent', 'programmed', 'hybrid')),
    room_state TEXT NOT NULL
        CHECK (room_state IN ('available', 'occupied', 'blocked', 'maintenance'))
);

CREATE TABLE system_users (
    id_user INTEGER PRIMARY KEY,
    username TEXT NOT NULL UNIQUE,
    password_hash TEXT NOT NULL,
    user_email TEXT NOT NULL UNIQUE,
    user_role TEXT NOT NULL
        CHECK (user_role IN (
            'surgeon',
            'head of surgery',
            'financial administrator',
            'scheduler'
        )),
    id_professional INTEGER UNIQUE,
    FOREIGN KEY (id_professional)
        REFERENCES professionals(id_professional)
);

CREATE TABLE procedures (
    id_procedure INTEGER PRIMARY KEY,
    procedure_code TEXT NOT NULL UNIQUE,
    procedure_name TEXT NOT NULL
);
""")

print("Master tables created successfully.")

Master tables created successfully.


In [ ]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)

display(tables)

,name
0,operating_rooms
1,procedures
2,professionals
3,system_users


###**Create the transaction tables**

In [ ]:
conn.executescript("""
CREATE TABLE surgical_operations (
    id_operation INTEGER PRIMARY KEY,

    patient_id TEXT NOT NULL,
    episode_id TEXT NOT NULL,

    id_room INTEGER,
    id_procedure INTEGER,

    actual_surgery_start TEXT,
    actual_surgery_end TEXT,

    actual_room_start TEXT,
    actual_room_end TEXT,

    status_so TEXT NOT NULL
        CHECK (status_so IN (
            'scheduled',
            'in_progress',
            'completed',
            'cancelled'
        )),

    FOREIGN KEY (id_room)
        REFERENCES operating_rooms(id_room),

    FOREIGN KEY (id_procedure)
        REFERENCES procedures(id_procedure),

    CHECK (
        actual_surgery_start IS NULL
        OR actual_surgery_end IS NULL
        OR actual_surgery_end > actual_surgery_start    --we check that if the surgery times are written down that the end time is bigger than the start time
    ),

    CHECK (
        actual_room_start IS NULL
        OR actual_room_end IS NULL
        OR actual_room_end > actual_room_start   --we check that if the room times are written down that the end time is bigger than the start time
    ),

    CHECK (
        status_so != 'completed'
        OR (
            actual_surgery_start IS NOT NULL
            AND actual_surgery_end IS NOT NULL
            AND actual_room_start IS NOT NULL
            AND actual_room_end IS NOT NULL    --by the time the operation is done we must know the room and surgery times
        )
    )
);
""")

print("Surgical operations table created successfully.")

Surgical operations table created successfully.


In [ ]:
conn.executescript("""
CREATE TABLE schedule (
    id_schedule INTEGER PRIMARY KEY,

    id_operation INTEGER NOT NULL,
    id_room INTEGER NOT NULL,

    scheduled_surgery_start TEXT NOT NULL,
    scheduled_surgery_end TEXT NOT NULL,

    scheduled_room_start TEXT NOT NULL,
    scheduled_room_end TEXT NOT NULL,

    status_s TEXT NOT NULL
        CHECK (status_s IN (
            'scheduled',
            'confirmed',
            'cancelled',
            'rescheduled',
            'completed'
        )),

    rescheduled_from INTEGER,

    FOREIGN KEY (id_operation)
        REFERENCES surgical_operations(id_operation),

    FOREIGN KEY (id_room)
        REFERENCES operating_rooms(id_room),

    FOREIGN KEY (rescheduled_from)
        REFERENCES schedule(id_schedule),

    CHECK (scheduled_surgery_end > scheduled_surgery_start),

    CHECK (scheduled_room_end > scheduled_room_start),

    CHECK (
        scheduled_room_start <= scheduled_surgery_start
        AND scheduled_room_end >= scheduled_surgery_end
    )
);
""")

print("Schedule table created successfully.")

Schedule table created successfully.


In [ ]:
conn.executescript("""
CREATE TABLE operation_team (
    id_operation_team INTEGER PRIMARY KEY,

    id_operation INTEGER NOT NULL,
    id_professional INTEGER NOT NULL,

    role_in_operation TEXT NOT NULL
        CHECK (role_in_operation IN (
            'surgeon',
            'assistant_surgeon',
            'anaesthesiologist',
            'nurse'
        )),

    FOREIGN KEY (id_operation)
        REFERENCES surgical_operations(id_operation),

    FOREIGN KEY (id_professional)
        REFERENCES professionals(id_professional),

    UNIQUE (id_operation, id_professional, role_in_operation)   -- this way we make sure that we don't have repeated operations, or the same professional doing the same role(we can have then a professional doing various roles)
);
""")

print("Operation team table created successfully.")

Operation team table created successfully.


In [ ]:
conn.executescript("""
CREATE TABLE operation_costs (
    id_operation_cost INTEGER PRIMARY KEY,

    id_operation INTEGER NOT NULL,

    cost_concept TEXT NOT NULL
        CHECK (cost_concept IN (
            'procedure',
            'operating room',
            'anaesthesia',
            'materials',
            'service'
        )),

    billing_unit TEXT NOT NULL
        CHECK (billing_unit IN (
            'fixed',
            'per minute',
            'per day'
        )),

    quantity NUMERIC NOT NULL
        CHECK (quantity > 0),

    unit_price NUMERIC NOT NULL
        CHECK (unit_price >= 0),

    FOREIGN KEY (id_operation)
        REFERENCES surgical_operations(id_operation)
);
""")

print("Operation costs table created successfully.")

Operation costs table created successfully.


In [ ]:
conn.executescript("""
CREATE TABLE surgical_complications (
    id_complication INTEGER PRIMARY KEY,

    id_operation INTEGER NOT NULL,

    complication_type TEXT NOT NULL,
    detection_date TEXT NOT NULL,
    description TEXT,

    severity TEXT
        CHECK (severity IN (
            'mild',
            'moderate',
            'severe',
            'critical'
        )),

    status_sc TEXT NOT NULL
        CHECK (status_sc IN (
            'active',
            'under treatment',
            'resolved'
        )),

    FOREIGN KEY (id_operation)
        REFERENCES surgical_operations(id_operation)
);
""")

print("Surgical complications table created successfully.")

Surgical complications table created successfully.


In [ ]:
conn.executescript("""
CREATE TABLE surgical_reports (
    id_report INTEGER PRIMARY KEY,

    id_operation INTEGER NOT NULL UNIQUE,

    created_at TEXT NOT NULL,
    current_version INTEGER,

    status_r TEXT NOT NULL
        CHECK (status_r IN (
            'draft',
            'completed',
            'approved'
        )),

    CHECK (
        current_version IS NULL
        OR current_version >= 1.  -- last version of the report
    ),

    FOREIGN KEY (id_operation)
        REFERENCES surgical_operations(id_operation)
);
""")

print("Surgical reports table created successfully.")

Surgical reports table created successfully.


In [ ]:
conn.executescript("""
CREATE TABLE surgical_report_versions (
    id_report_version INTEGER PRIMARY KEY,

    id_report INTEGER NOT NULL,
    version_number INTEGER NOT NULL,

    id_professional INTEGER NOT NULL,

    report_date TEXT NOT NULL,
    report_content TEXT NOT NULL,

    FOREIGN KEY (id_report)
        REFERENCES surgical_reports(id_report),

    FOREIGN KEY (id_professional)
        REFERENCES professionals(id_professional),

    CHECK (version_number >= 1),

    UNIQUE (id_report, version_number)
);
""")

print("Surgical report versions table created successfully.")

Surgical report versions table created successfully.


In [ ]:
conn.executescript("""
CREATE TABLE operation_images (
    id_image INTEGER PRIMARY KEY,

    id_operation INTEGER NOT NULL,
    id_complication INTEGER,
    id_professional INTEGER NOT NULL,

    image_path TEXT NOT NULL,
    image_date TEXT NOT NULL,
    upload_date TEXT NOT NULL,

    description TEXT,

    file_type TEXT NOT NULL
        CHECK (file_type IN (
            'jpg',
            'jpeg',
            'png'
        )),

    FOREIGN KEY (id_operation)
        REFERENCES surgical_operations(id_operation),

    FOREIGN KEY (id_complication)
        REFERENCES surgical_complications(id_complication),

    FOREIGN KEY (id_professional)
        REFERENCES professionals(id_professional)
);
""")

print("Operation images table created successfully.")

Operation images table created successfully.


In [ ]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
AND name NOT LIKE 'sqlite_%'
ORDER BY name;
""", conn)

display(tables)

print("Total number of tables:", len(tables))

,name
0,operating_rooms
1,operation_costs
2,operation_images
3,operation_team
4,procedures
5,professionals
6,schedule
7,surgical_complications
8,surgical_operations
9,surgical_report_versions


Total number of tables: 12


In [ ]:
for table in tables["name"]:
    print(f"\nTABLE: {table.upper()}")

    columns = pd.read_sql_query(
        f"PRAGMA table_info('{table}');",
        conn
    )

    display(columns[["name", "type", "notnull", "pk"]])


TABLE: OPERATING_ROOMS


,name,type,notnull,pk
0,id_room,INTEGER,0,1
1,room_code,TEXT,1,0
2,room_name,TEXT,1,0
3,room_location,TEXT,1,0
4,room_type,TEXT,1,0
5,room_state,TEXT,1,0



TABLE: OPERATION_COSTS


,name,type,notnull,pk
0,id_operation_cost,INTEGER,0,1
1,id_operation,INTEGER,1,0
2,cost_concept,TEXT,1,0
3,billing_unit,TEXT,1,0
4,quantity,NUMERIC,1,0
5,unit_price,NUMERIC,1,0



TABLE: OPERATION_IMAGES


,name,type,notnull,pk
0,id_image,INTEGER,0,1
1,id_operation,INTEGER,1,0
2,id_complication,INTEGER,0,0
3,id_professional,INTEGER,1,0
4,image_path,TEXT,1,0
5,image_date,TEXT,1,0
6,upload_date,TEXT,1,0
7,description,TEXT,0,0
8,file_type,TEXT,1,0


... (14 outputs left out)


TABLE: SURGICAL_REPORTS


,name,type,notnull,pk
0,id_report,INTEGER,0,1
1,id_operation,INTEGER,1,0
2,created_at,TEXT,1,0
3,current_version,INTEGER,0,0
4,status_r,TEXT,1,0



TABLE: SYSTEM_USERS


,name,type,notnull,pk
0,id_user,INTEGER,0,1
1,username,TEXT,1,0
2,password_hash,TEXT,1,0
3,user_email,TEXT,1,0
4,user_role,TEXT,1,0
5,id_professional,INTEGER,0,0


###**Insert test data (creation of dummy data)**

In [ ]:
conn.execute("""
INSERT INTO professionals (
    id_professional,
    professional_name,
    professional_type,
    professional_category,
    professional_email,
    professional_phone
)
VALUES
    (1, 'Alex Morgan', 'surgery', 'Specialist',
     'alex.morgan@example.com', '+34111111111'),

    (2, 'Taylor Smith', 'surgery', 'Consultant',
     'taylor.smith@example.com', '+34222222222'),

    (3, 'Jordan Lee', 'anaesthesia', 'Specialist',
     'jordan.lee@example.com', '+34333333333'),

    (4, 'Casey Brown', 'nursing', 'Resident',
     'casey.brown@example.com', '+34444444444'),

    (5, 'Riley Davis', 'nursing', 'Specialist',
     'riley.davis@example.com', '+34555555555');
""")

conn.commit()

print("Professional test data inserted successfully.")

Professional test data inserted successfully.


In [ ]:
conn.execute("""
INSERT INTO operating_rooms (
    id_room, room_code, room_name,
    room_location, room_type, room_state
)
VALUES
    (1, 'OR-01', 'Emergency Operating Room',
     'Floor 2 - East Wing', 'urgent', 'available'),

    (2, 'OR-02', 'General Surgery Room',
     'Floor 2 - West Wing', 'programmed', 'available'),

    (3, 'OR-03', 'Hybrid Operating Room',
     'Floor 3 - East Wing', 'hybrid', 'available');
""")

conn.commit()

In [ ]:
conn.execute("""
INSERT INTO system_users (
    id_user, username, password_hash,
    user_email, user_role, id_professional
)
VALUES
    (1, 'alex_morgan', 'demo_hash_001',
     'alex.morgan@example.com', 'surgeon', 1),

    (2, 'taylor_smith', 'demo_hash_002',
     'taylor.smith@example.com', 'head of surgery', 2),

    (3, 'finance_admin', 'demo_hash_003',
     'finance@example.com', 'financial administrator', NULL),

    (4, 'scheduler_user', 'demo_hash_004',
     'scheduler@example.com', 'scheduler', NULL);
""")

conn.commit()
print("System user test data inserted successfully.")

System user test data inserted successfully.


In [ ]:
conn.execute("""
INSERT INTO procedures (
    id_procedure, procedure_code, procedure_name
)
VALUES
    (1, '0DTJ0ZZ', 'Resection of Appendix, Open Approach'),
    (2, '0DTJ4ZZ', 'Resection of Appendix, Percutaneous Endoscopic Approach'); -- ICD -10-PCS, this is coding referring to the type of procedure and how it's done, we put this one as an example, but we need the teacher to give us the defenitive ones
""")

conn.commit()
print("Procedure test data inserted successfully.")

Procedure test data inserted successfully.


In [ ]:
conn.execute("""
INSERT INTO surgical_operations (
    id_operation,
    patient_id,
    episode_id,
    id_room,
    id_procedure,
    actual_surgery_start,
    actual_surgery_end,
    actual_room_start,
    actual_room_end,
    status_so
)
VALUES
    (1, 1001, 5001, 1, 2,     -- we invent how the patient_id and episode_id looklike till we contact with group A
     '2026-11-10 09:00:00',
     '2026-11-10 10:00:00',
     '2026-11-10 08:30:00',
     '2026-11-10 10:30:00',
     'completed'),

    (2, 1002, 5002, 2, 2,
     NULL, NULL, NULL, NULL,   -- we see that until the operation is not completed we cannot know the actual values
     'scheduled'),

    (3, 1003, 5003, 2, 1,
     NULL, NULL, NULL, NULL,
     'scheduled');
""")

conn.commit()
print("Surgical operation test data inserted successfully.")

Surgical operation test data inserted successfully.


In [ ]:
conn.execute("""
INSERT INTO schedule (
    id_schedule,
    id_operation,
    id_room,
    scheduled_surgery_start,
    scheduled_surgery_end,
    scheduled_room_start,
    scheduled_room_end,
    status_s,
    rescheduled_from
)
VALUES
    (1, 1, 1,
     '2026-11-10 09:00:00',
     '2026-11-10 10:00:00',
     '2026-11-10 08:30:00',
     '2026-11-10 10:30:00',
     'completed',
     NULL),

    (2, 2, 2,
     '2026-11-17 09:00:00',
     '2026-11-17 10:00:00',
     '2026-11-17 08:30:00',
     '2026-11-17 10:30:00',
     'confirmed',
     NULL),

    (3, 3, 2,
     '2026-11-17 09:30:00',
     '2026-11-17 10:30:00',
     '2026-11-17 09:00:00',
     '2026-11-17 11:00:00',
     'scheduled',
     NULL);
""")

conn.commit()
print("Schedule test data inserted successfully.")

Schedule test data inserted successfully.


In [ ]:
conn.execute("""
INSERT INTO operation_team (
    id_operation_team,
    id_operation,
    id_professional,
    role_in_operation
)
VALUES
    (1, 1, 1, 'surgeon'),
    (2, 1, 3, 'anaesthesiologist'),
    (3, 1, 4, 'nurse'),
    (4, 2, 2, 'surgeon'),
    (5, 2, 3, 'anaesthesiologist'),
    (6, 3, 1, 'surgeon'),
    (7, 3, 5, 'nurse');
""")

conn.commit()
print("Operation team test data inserted successfully.")

Operation team test data inserted successfully.


In [ ]:
conn.execute("""
INSERT INTO operation_costs (
    id_operation_cost,
    id_operation,
    cost_concept,
    billing_unit,
    quantity,
    unit_price
)
VALUES
    (1, 1, 'procedure', 'fixed', 1, 850.00),
    (2, 1, 'operating room', 'per minute', 120, 8.50),
    (3, 1, 'anaesthesia', 'fixed', 1, 250.00),
    (4, 1, 'materials', 'fixed', 1, 180.00),
    (5, 2, 'procedure', 'fixed', 1, 850.00),
    (6, 3, 'procedure', 'fixed', 1, 850.00);
""")

conn.commit()
print("Operation cost test data inserted successfully.")

Operation cost test data inserted successfully.


In [ ]:
conn.execute("""
INSERT INTO surgical_complications (
    id_complication,
    id_operation,
    complication_type,
    detection_date,
    description,
    severity,
    status_sc
)
VALUES
    (1, 1,
     'Postoperative infection',
     '2026-11-16 09:00:00',
     'Fictional example of an infection detected six days after surgery.',
     'moderate',
     'under treatment');
""")

conn.commit()
print("Surgical complication test data inserted successfully.")

Surgical complication test data inserted successfully.


In [ ]:
conn.execute("""
INSERT INTO surgical_reports (
    id_report,
    id_operation,
    created_at,
    status_r
)
VALUES
    (1, 1, '2026-11-10 11:00:00', 'approved');
""")

conn.commit()
print("Surgical report test data inserted successfully.")

Surgical report test data inserted successfully.


In [ ]:
conn.execute("""
INSERT INTO surgical_report_versions (
    id_report_version,
    id_report,
    version_number,
    id_professional,
    report_date,
    report_content
)
VALUES
    (1, 1, 1, 1,
     '2026-11-10 11:00:00',
     'Initial fictional surgical report: appendectomy completed successfully.'),

    (2, 1, 2, 2,
     '2026-11-10 12:00:00',
     'Updated fictional surgical report with additional clinical documentation.');
""")

conn.commit()
print("Surgical report version test data inserted successfully.")

Surgical report version test data inserted successfully.


In [ ]:
conn.execute("""
INSERT INTO operation_images (
    id_image,
    id_operation,
    id_complication,
    id_professional,
    image_path,
    image_date,
    upload_date,
    description,
    file_type
)
VALUES
    (1, 1, 1, 4,
     'test_images/operation_1_day_6_01.jpg',
     '2026-11-16 09:00:00',
     '2026-11-16 09:15:00',
     'Fictional postoperative image reference 1.',
     'jpg'),

    (2, 1, 1, 4,
     'test_images/operation_1_day_6_02.png',
     '2026-11-16 09:05:00',
     '2026-11-16 09:20:00',
     'Fictional postoperative image reference 2.',
     'png');
""")

conn.commit()
print("Operation image test data inserted successfully.")

Operation image test data inserted successfully.


###**Check the test data**

In [ ]:
tables = pd.read_sql_query("""  -- to make sure all the information is saved
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)

for table in tables["name"]:
    print(f"\n--- {table.upper()} ---")
    data = pd.read_sql_query(
        f'SELECT * FROM "{table}";', conn
    )
    print("Number of records:", len(data))
    display(data)


--- OPERATING_ROOMS ---
Number of records: 3


,id_room,room_code,room_name,room_location,room_type,room_state
0,1,OR-01,Emergency Operating Room,Floor 2 - East Wing,urgent,available
1,2,OR-02,General Surgery Room,Floor 2 - West Wing,programmed,available
2,3,OR-03,Hybrid Operating Room,Floor 3 - East Wing,hybrid,available



--- OPERATION_COSTS ---
Number of records: 6


,id_operation_cost,id_operation,cost_concept,billing_unit,quantity,unit_price
0,1,1,procedure,fixed,1,850.0
1,2,1,operating room,per minute,120,8.5
2,3,1,anaesthesia,fixed,1,250.0
3,4,1,materials,fixed,1,180.0
4,5,2,procedure,fixed,1,850.0
5,6,3,procedure,fixed,1,850.0



--- OPERATION_IMAGES ---
Number of records: 2


,id_image,id_operation,id_complication,id_professional,image_path,image_date,upload_date,description,file_type
0,1,1,1,4,test_images/operation_1_day_6_01.jpg,2026-11-16 09:00:00,2026-11-16 09:15:00,Fictional postoperative image reference 1.,jpg
1,2,1,1,4,test_images/operation_1_day_6_02.png,2026-11-16 09:05:00,2026-11-16 09:20:00,Fictional postoperative image reference 2.,png


... (14 outputs left out)


--- SURGICAL_REPORTS ---
Number of records: 1


,id_report,id_operation,created_at,current_version,status_r
0,1,1,2026-11-10 11:00:00,None,approved



--- SYSTEM_USERS ---
Number of records: 4


,id_user,username,password_hash,user_email,user_role,id_professional
0,1,alex_morgan,demo_hash_001,alex.morgan@example.com,surgeon,1.0
1,2,taylor_smith,demo_hash_002,taylor.smith@example.com,head of surgery,2.0
2,3,finance_admin,demo_hash_003,finance@example.com,financial administrator,NaN
3,4,scheduler_user,demo_hash_004,scheduler@example.com,scheduler,NaN


In [ ]:
foreign_key_errors = pd.read_sql_query(""" --If an empty table appears, it means that SQLite has not found any referential integrity errors.
PRAGMA foreign_key_check;
""", conn)

display(foreign_key_errors)

,table,rowid,parent,fkid


In [ ]:
conflicts = pd.read_sql_query("""  -- to see if it can detect conflicts
SELECT
    s1.id_schedule AS operation_1,
    s2.id_schedule AS operation_2,
    s1.id_room AS operating_room,
    s1.scheduled_room_start AS start_1,
    s1.scheduled_room_end AS end_1,
    s2.scheduled_room_start AS start_2,
    s2.scheduled_room_end AS end_2
FROM schedule s1
JOIN schedule s2
    ON s1.id_room = s2.id_room -- we are looking for diferent operations taking the same room
    AND s1.id_schedule < s2.id_schedule
    AND s1.status_s IN ('scheduled', 'confirmed')
    AND s2.status_s IN ('scheduled', 'confirmed')
    AND s1.scheduled_room_start < s2.scheduled_room_end
    AND s2.scheduled_room_start < s1.scheduled_room_end; -- if we schedule a second operation and this one beggings before the first one ends.
""", conn)

display(conflicts)

,operation_1,operation_2,operating_room,start_1,end_1,start_2,end_2
0,2,3,2,2026-11-17 08:30:00,2026-11-17 10:30:00,2026-11-17 09:00:00,2026-11-17 11:00:00


In [ ]:
operations_test = pd.read_sql_query(""" -- to make sure that the operations are correctly linked to the operating rooms and the procedures.
SELECT
    so.id_operation,
    so.patient_id,
    r.room_name,
    p.procedure_name
FROM surgical_operations so
LEFT JOIN operating_rooms r
    ON so.id_room = r.id_room
LEFT JOIN procedures p
    ON so.id_procedure = p.id_procedure;
""", conn)

display(operations_test)

,id_operation,patient_id,room_name,procedure_name
0,1,1001,Emergency Operating Room,"Resection of Appendix, Percutaneous Endoscopic..."
1,2,1002,General Surgery Room,"Resection of Appendix, Percutaneous Endoscopic..."
2,3,1003,General Surgery Room,"Resection of Appendix, Open Approach"


In [ ]:
costs_test = pd.read_sql_query("""  --to obtain total costs
SELECT
    id_operation,
    SUM(quantity * unit_price) AS total_cost
FROM operation_costs
GROUP BY id_operation;
""", conn)

display(costs_test)

,id_operation,total_cost
0,1,2300.0
1,2,850.0
2,3,850.0


In [ ]:
team_test = pd.read_sql_query(""" --check the surgical equipment
SELECT
    ot.id_operation,
    p.professional_name,
    ot.role_in_operation
FROM operation_team ot
JOIN professionals p
    ON ot.id_professional = p.id_professional;
""", conn)

display(team_test)

,id_operation,professional_name,role_in_operation
0,1,Alex Morgan,surgeon
1,1,Jordan Lee,anaesthesiologist
2,1,Casey Brown,nurse
3,2,Taylor Smith,surgeon
4,2,Jordan Lee,anaesthesiologist
5,3,Alex Morgan,surgeon
6,3,Riley Davis,nurse


In [ ]:
reports_test = pd.read_sql_query("""  --check reports and complications
SELECT
    so.id_operation,
    sr.id_report,
    sc.complication_type,
    sc.status_sc AS complication_status
FROM surgical_operations so
LEFT JOIN surgical_reports sr
    ON so.id_operation = sr.id_operation
LEFT JOIN surgical_complications sc
    ON so.id_operation = sc.id_operation;
""", conn)

display(reports_test)

,id_operation,id_report,complication_type,complication_status
0,1,1.0,Postoperative infection,under treatment
1,2,NaN,None,None
2,3,NaN,None,None
